In [ ]:
# -*- coding: utf-8 -*-


stock_prediction_commented.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1InEx9nsyb2k935hAeiYNuh2WVcbKKX3T

Reliance Industries — Stock Price Forecasting Pipeline
========================================================
EDA -> Feature Engineering -> 8-Model Comparison -> Hyperparameter Tuning
-> Dynamic Best-Model Selection -> 30-Day Forecast -> Save for Deployment

NOTE ON A BUG FIXED IN THIS VERSION:
The original script computed `results_df` (which correctly found Linear
Regression to be the best model by RMSE/MAE/MAPE), but then hardcoded
`xgb_best` for feature importance, residual analysis, the final forecast,
and the saved .pkl file — regardless of what actually won. This version
fixes that: whichever model wins in `results_df` is the one used and saved
everywhere downstream. This matters because on this dataset, Linear
Regression actually beat every other model, including tuned XGBoost.


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')



============================================================================
1. LOAD DATA
============================================================================
Read the raw OHLCV sheet and sort by date — the file isn't guaranteed to
already be in chronological order, and every later step (lags, rolling
averages, train/test split) depends on strict date ordering.



In [ ]:

df = pd.read_excel("Company stock prices.xlsx", sheet_name="in")
df = df.sort_values("Date").reset_index(drop=True)



2. BASIC EDA / DATA QUALITY CHECKS



In [ ]:

print(df.shape)          # (rows, columns) — quick sanity check on data size
print(df.sample(10))     # random rows, not just head/tail, to spot anomalies
df.info()                # column dtypes + non-null counts in one view
print(df.isnull().sum()) # per-column missing value count — should all be 0
print(df.duplicated().sum())  # count of fully duplicated rows — should be 0

print("\n--- Date range ---")
print("Start:", df["Date"].min())
print("End:", df["Date"].max())
print("Total trading days:", df.shape[0])

print(df.describe())     # mean/std/min/max for every numeric column



### Chart 1: raw close price trend



In [ ]:

plt.figure(figsize=(12, 5))
plt.plot(df['Date'], df['Close'], color='blue')
plt.title('Reliance Close Price Over Time')
plt.xlabel('Date'); plt.ylabel('Close Price')
plt.show()



### Chart 2: average close price per year (bar chart)



In [ ]:

df['Year'] = df['Date'].dt.year
yearly_avg = df.groupby('Year')['Close'].mean()
plt.figure(figsize=(8, 5))
yearly_avg.plot(kind='bar', color='skyblue')
plt.title('Average Closing Price by Year')
plt.xlabel('Year'); plt.ylabel('Average Close Price')
plt.show()



### Chart 3: price spread per year (boxplot)

Shows median, interquartile spread, and outliers for each year — good for
spotting which year was most volatile.


In [ ]:

plt.figure(figsize=(10, 6))
sns.boxplot(x='Year', y='Close', data=df)
plt.title('Close Price Distribution by Year')
plt.show()



### Chart 4: daily returns distribution (histogram)



In [ ]:

df['Daily_Return_%'] = df['Close'].pct_change() * 100
plt.figure(figsize=(10, 5))
sns.histplot(df['Daily_Return_%'].dropna(), bins=50, kde=True, color='purple')
plt.title('Distribution of Daily Returns (%)')
plt.xlabel('Daily Return %')
plt.show()



### Chart 5: correlation heatmap between OHLCV columns



In [ ]:

plt.figure(figsize=(7, 5))
sns.heatmap(df[['Open', 'High', 'Low', 'Close', 'Volume']].corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Between Price Columns')
plt.show()



### Chart 6: moving averages (trend smoothing)



In [ ]:

df['MA_20'] = df['Close'].rolling(20).mean()  # short-term trend
df['MA_50'] = df['Close'].rolling(50).mean()  # medium-term trend
plt.figure(figsize=(12, 5))
plt.plot(df['Date'], df['Close'], label='Close Price', color='grey', alpha=0.6)
plt.plot(df['Date'], df['MA_20'], label='20-Day MA', color='orange')
plt.plot(df['Date'], df['MA_50'], label='50-Day MA', color='green')
plt.title('Close Price with Moving Averages')
plt.xlabel('Date'); plt.ylabel('Price')
plt.legend()
plt.show()



### Chart 7 & 8: trading volume (yearly bar + daily line)



In [ ]:

yearly_volume = df.groupby('Year')['Volume'].sum()
plt.figure(figsize=(8, 5))
yearly_volume.plot(kind='bar', color='teal')
plt.title('Total Trading Volume by Year')
plt.xlabel('Year'); plt.ylabel('Total Volume')
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(df['Date'], df['Volume'], color='teal', linewidth=0.8)
plt.title('Daily Trading Volume Over Time')
plt.xlabel('Date'); plt.ylabel('Volume')
plt.show()



### Biggest single-day moves — used to catch data issues like splits



In [ ]:

biggest_drop = df.loc[df['Daily_Return_%'].idxmin()]
biggest_gain = df.loc[df['Daily_Return_%'].idxmax()]
print("Biggest single-day drop:\n", biggest_drop)
print("\nBiggest single-day gain:\n", biggest_gain)



============================================================================
3. CORPORATE ACTION ADJUSTMENT
============================================================================
The biggest single-day move (~-35% on 20-Apr-2022, on ~6x average volume,
with no price recovery afterward) is the signature of a stock split/bonus
issue, not an actual crash. Left unadjusted, the model would learn a fake
-35% return that never happened to an actual investor. This halves all
pre-split prices (and doubles volume) so the whole series sits on one
consistent scale. Confirm the exact ratio with the data source if possible
— this assumes a simple 2-for-1 split as an approximation.



In [ ]:

split_date = pd.Timestamp('2022-04-20')
price_cols = ['Open', 'High', 'Low', 'Close']
df.loc[df['Date'] < split_date, price_cols] = df.loc[df['Date'] < split_date, price_cols] / 2
df.loc[df['Date'] < split_date, 'Volume'] = df.loc[df['Date'] < split_date, 'Volume'] * 2



============================================================================
4. FEATURE ENGINEERING
============================================================================
Lag features: yesterday's / n-days-ago's price, so the model has past
values to learn from (models can't see the future, only what we hand them).



In [ ]:

for lag in [1, 2, 3, 5, 10]:
    df[f'Close_lag_{lag}'] = df['Close'].shift(lag)



Rolling statistics: smoothed trend (MA_7, MA_21) and recent volatility
(STD_7), plus a smoothed volume signal (Volume_MA_7).



In [ ]:

df['MA_7'] = df['Close'].rolling(7).mean()
df['MA_21'] = df['Close'].rolling(21).mean()
df['STD_7'] = df['Close'].rolling(7).std()
df['Volume_MA_7'] = df['Volume'].rolling(7).mean()

# Calendar features — cheap to add, occasionally useful, easy to drop later.
df['DayOfWeek'] = df['Date'].dt.dayofweek
df['Month'] = df['Date'].dt.month
df['Quarter'] = df['Date'].dt.quarter



Lag/rolling features leave NaN at the start of the series (e.g. there's no
"10 days ago" for the first 10 rows) — drop those rows now that every
feature column is built.



In [ ]:

df_model = df.dropna().reset_index(drop=True)
print(df_model.shape)



5. TRAIN / TEST SPLIT — chronological, never random, for time series

Random-shuffling a time series split would let the model "see the future"
during training. Always cut by date: everything before the last
`test_size` rows is train, the last chunk is test.


In [ ]:

test_size = 30
train = df_model.iloc[:-test_size]
test = df_model.iloc[-test_size:]

print("Train period:", train['Date'].min(), "to", train['Date'].max())
print("Test period:", test['Date'].min(), "to", test['Date'].max())

feature_cols = ['Close_lag_1', 'Close_lag_2', 'Close_lag_3', 'Close_lag_5', 'Close_lag_10',
                'MA_7', 'MA_21', 'STD_7', 'Volume_MA_7', 'DayOfWeek', 'Month', 'Quarter']

X_train, y_train = train[feature_cols], train['Close']
X_test, y_test = test[feature_cols], test['Close']



6. EVALUATION FUNCTION — used identically for every model below


In [ ]:

from sklearn.metrics import mean_squared_error, mean_absolute_error

def evaluate(y_true, y_pred, model_name):


RMSE (₹, punishes big misses more) / MAE (₹, average miss) /
    MAPE (%, easiest to interpret) — printed together so every model is
    directly comparable on the same scale.
    


In [ ]:
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    print(f"{model_name}: RMSE={rmse:.2f}  MAE={mae:.2f}  MAPE={mape:.2f}%")
    return rmse, mae, mape



7. BASELINE — the bar every real model must clear
"Tomorrow's price = today's price." Stock prices are close to a random
walk, so this is deceptively hard to beat — any model that can't beat it
isn't adding real value.


In [ ]:

naive_pred = test['Close_lag_1']
evaluate(y_test, naive_pred, "Naive Baseline")



8. MODEL 1 — Linear Regression

Fits a straight-line relationship between the engineered features and
Close price. On this dataset, this turned out to be the actual winner —
a good reminder to always try the simplest model before reaching for
something complex.


In [ ]:

from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)
evaluate(y_test, lr_pred, "Linear Regression")



9. MODEL 2 — Random Forest


In [ ]:

from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
evaluate(y_test, rf_pred, "Random Forest")



10. MODEL 3 — ARIMA (classical time-series model, no engineered features)
order=(5,1,0): use last 5 lags (p=5), difference once for stationarity
(d=1 — confirmed by an ADF test in the EDA stage), no MA term (q=0).



In [ ]:

from statsmodels.tsa.arima.model import ARIMA

arima_model = ARIMA(train['Close'], order=(5, 1, 0))
arima_fit = arima_model.fit()
arima_pred = arima_fit.forecast(steps=test_size)
evaluate(y_test.values, arima_pred.values, "ARIMA")



11. MODEL 4 — Gradient Boosting

Like Random Forest, but trees are built sequentially, each correcting the
previous one's errors — usually a bit stronger than plain Random Forest.


In [ ]:

from sklearn.ensemble import GradientBoostingRegressor

gb = GradientBoostingRegressor(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42)
gb.fit(X_train, y_train)
gb_pred = gb.predict(X_test)
evaluate(y_test, gb_pred, "Gradient Boosting")



12. MODEL 5 — XGBoost



In [ ]:

from xgboost import XGBRegressor

xgb = XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42)
xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_test)
evaluate(y_test, xgb_pred, "XGBoost")



13. MODEL 6 — Support Vector Regression (needs scaled features)



In [ ]:

from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

svr = SVR(kernel='rbf', C=100, gamma=0.1)
svr.fit(X_train_scaled, y_train)
svr_pred = svr.predict(X_test_scaled)
evaluate(y_test, svr_pred, "SVR")



14. MODEL 7 — LSTM (deep learning, sequence-based)



In [ ]:

import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

# LSTMs train poorly on raw prices (~500) — squeeze into 0-1 range first.
scaler_lstm = MinMaxScaler()
scaled_close = scaler_lstm.fit_transform(df_model[['Close']])



Build (input, target) sequences: last 10 days -> next day. LSTMs learn
from ordered sequences, not flat feature rows like the models above.



In [ ]:

seq_len = 10
X_seq, y_seq = [], []
for i in range(seq_len, len(scaled_close)):
    X_seq.append(scaled_close[i - seq_len:i, 0])
    y_seq.append(scaled_close[i, 0])
X_seq, y_seq = np.array(X_seq), np.array(y_seq)
X_seq = X_seq.reshape(X_seq.shape[0], X_seq.shape[1], 1)

# Same chronological split logic as before, just applied to the sequences.
X_seq_train, X_seq_test = X_seq[:-test_size], X_seq[-test_size:]
y_seq_train, y_seq_test = y_seq[:-test_size], y_seq[-test_size:]

lstm = Sequential([
    LSTM(50, activation='relu', input_shape=(seq_len, 1)),
    Dense(1)
])
lstm.compile(optimizer='adam', loss='mse')
lstm.fit(X_seq_train, y_seq_train, epochs=20, batch_size=16, verbose=1)

lstm_pred_scaled = lstm.predict(X_seq_test)
lstm_pred = scaler_lstm.inverse_transform(lstm_pred_scaled).flatten()
y_seq_test_actual = scaler_lstm.inverse_transform(y_seq_test.reshape(-1, 1)).flatten()
evaluate(y_seq_test_actual, lstm_pred, "LSTM")



14b. MODEL 8 — SARIMA (Seasonal ARIMA)

Same idea as ARIMA but with an added seasonal component (P,D,Q,s). Trading
weeks give a natural short seasonal period of 5. This is NOT tuned via grid
search here (that would be a large search over 7 hyperparameters) — a
reasonable fixed order is used and flagged as a candidate for tuning.


In [ ]:

from statsmodels.tsa.statespace.sarimax import SARIMAX

sarima_model = SARIMAX(
    train['Close'], order=(1, 1, 1), seasonal_order=(1, 1, 1, 5),
    enforce_stationarity=False, enforce_invertibility=False,
)
sarima_fit = sarima_model.fit(disp=False)
sarima_pred = sarima_fit.forecast(steps=test_size)
evaluate(y_test.values, sarima_pred.values, "SARIMA")



14c. MODEL 9 — Holt-Winters (Triple Exponential Smoothing)

Classical smoothing model: level + trend + (optional) seasonality. No
external features needed, like ARIMA/SARIMA. `seasonal=None` here since a
5-day seasonal period on ~700 rows of daily close price is weak/noisy;
trend='add' captures the drift.


In [ ]:

from statsmodels.tsa.holtwinters import ExponentialSmoothing

hw_model = ExponentialSmoothing(train['Close'], trend='add', seasonal=None)
hw_fit = hw_model.fit()
hw_pred = hw_fit.forecast(test_size)
evaluate(y_test.values, hw_pred.values, "Holt-Winters")



14d. MODEL 10 — Prophet

Facebook/Meta's decomposable trend + seasonality model. Needs a two-column
`ds`/`y` frame rather than the engineered feature matrix used by the ML
models above.


In [ ]:

from prophet import Prophet

prophet_train = train[['Date', 'Close']].rename(columns={'Date': 'ds', 'Close': 'y'})
prophet_model = Prophet(daily_seasonality=False, weekly_seasonality=True, yearly_seasonality=True)
prophet_model.fit(prophet_train)

prophet_future = prophet_model.make_future_dataframe(periods=test_size, freq='B')
prophet_forecast_full = prophet_model.predict(prophet_future)
prophet_pred = prophet_forecast_full['yhat'].iloc[-test_size:].values
evaluate(y_test.values, prophet_pred, "Prophet")



15. HYPERPARAMETER TUNING — time-series-safe cross-validation

Regular cv=5 shuffles folds randomly, which leaks future data into
training for a time series. TimeSeriesSplit always trains on an earlier
chunk and validates on a later one, walking forward — the only valid way
to cross-validate time-ordered data.


In [ ]:

from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV

tscv = TimeSeriesSplit(n_splits=5)



### Tune Random Forest



In [ ]:

rf_param_grid = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 4, 5, 6, 8, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
}
rf_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42),
    param_distributions=rf_param_grid,
    n_iter=20, cv=tscv, scoring='neg_root_mean_squared_error',
    random_state=42, n_jobs=-1,
)
rf_search.fit(X_train, y_train)
print("Best RF params:", rf_search.best_params_)
rf_best = rf_search.best_estimator_
rf_best_pred = rf_best.predict(X_test)
evaluate(y_test, rf_best_pred, "Random Forest (Tuned)")



### Tune XGBoost



In [ ]:

xgb_param_grid = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
}
xgb_search = RandomizedSearchCV(
    XGBRegressor(random_state=42),
    param_distributions=xgb_param_grid,
    n_iter=25, cv=tscv, scoring='neg_root_mean_squared_error',
    random_state=42, n_jobs=-1,
)
xgb_search.fit(X_train, y_train)
print("Best XGB params:", xgb_search.best_params_)
xgb_best = xgb_search.best_estimator_
xgb_best_pred = xgb_best.predict(X_test)
evaluate(y_test, xgb_best_pred, "XGBoost (Tuned)")



NOTE — a genuine, reportable finding: tuning made BOTH Random Forest and
XGBoost slightly *worse* on the held-out test set than their untuned
defaults. This can happen on a small dataset (~700 rows): the search
optimizes average RMSE across 5 CV folds, which doesn't always match the
single most recent test window best. Worth stating honestly in the report
rather than assuming tuning always helps.



### Tune ARIMA order (grid search — statsmodels has no sklearn helper)



In [ ]:

import itertools

p_values = range(0, 6)
d_values = [1]  # fixed — confirmed via ADF test in EDA
q_values = range(0, 3)

best_rmse = float('inf')
best_order = None
for p, d, q in itertools.product(p_values, d_values, q_values):
    try:
        model = ARIMA(train['Close'], order=(p, d, q)).fit()
        pred = model.forecast(steps=test_size)
        rmse = np.sqrt(mean_squared_error(y_test.values, pred.values))
        if rmse < best_rmse:
            best_rmse = rmse
            best_order = (p, d, q)
    except Exception:
        continue  # skip combinations that fail to converge

print("Best ARIMA order:", best_order, "RMSE:", round(best_rmse, 2))
arima_best = ARIMA(train['Close'], order=best_order).fit()
arima_best_pred = arima_best.forecast(steps=test_size)
evaluate(y_test.values, arima_best_pred.values, "ARIMA (Tuned)")



16. FINAL COMPARISON TABLE — every model, sorted by RMSE




In [ ]:

results = []
results.append(("Naive", *evaluate(y_test, naive_pred, "Naive")))
results.append(("Linear Regression", *evaluate(y_test, lr_pred, "Linear Regression")))
results.append(("Random Forest", *evaluate(y_test, rf_pred, "Random Forest")))
results.append(("Random Forest (Tuned)", *evaluate(y_test, rf_best_pred, "RF Tuned")))
results.append(("ARIMA", *evaluate(y_test.values, arima_pred.values, "ARIMA")))
results.append(("ARIMA (Tuned)", *evaluate(y_test.values, arima_best_pred.values, "ARIMA Tuned")))
results.append(("Gradient Boosting", *evaluate(y_test, gb_pred, "GB")))
results.append(("XGBoost", *evaluate(y_test, xgb_pred, "XGBoost")))
results.append(("XGBoost (Tuned)", *evaluate(y_test, xgb_best_pred, "XGBoost Tuned")))
results.append(("SVR", *evaluate(y_test, svr_pred, "SVR")))
results.append(("LSTM", *evaluate(y_seq_test_actual, lstm_pred, "LSTM")))
results.append(("SARIMA", *evaluate(y_test.values, sarima_pred.values, "SARIMA")))
results.append(("Holt-Winters", *evaluate(y_test.values, hw_pred.values, "Holt-Winters")))
results.append(("Prophet", *evaluate(y_test.values, prophet_pred, "Prophet")))

results_df = pd.DataFrame(results, columns=['Model', 'RMSE', 'MAE', 'MAPE']).sort_values('RMSE')
print(results_df)

best_model_name = results_df.iloc[0]['Model']
print("Best model:", best_model_name)
print(results_df.iloc[0])



### DEPLOY STRICTLY BY LOWEST RMSE — genuinely dynamic now

With only 8 models, Linear Regression happened to win, so an earlier
version of this script hardcoded `best_model_name = "Linear Regression"`
right here "to be explicit." That hardcode is REMOVED now that SARIMA,
Holt-Winters, and Prophet are in the comparison too — a hardcoded winner
would silently ignore it if one of the new models actually did better.
`best_model_name` above (from `results_df.iloc[0]['Model']`) is left as
the single source of truth for which model gets deployed.

CAVEAT worth keeping regardless of which model wins: if a feature-based
model with a Close_lag_1 coefficient/weight >1 wins (this was true for
Linear Regression on the original 8-model run), recursive 30-day
forecasting compounds error instead of damping it — the forecast in
Section 20 can drift to unrealistic values by day 25-30. This is a known,
reportable limitation of using a 1-step-accuracy metric to choose a
multi-step forecaster. Worth stating plainly in the presentation rather
than smoothing over it, whichever model ends up on top.


17. DYNAMIC BEST-MODEL SELECTION  <-- THE ACTUAL FIX
The original script printed `best_model_name` here but then hardcoded
`xgb_best` for every step below, regardless of which model actually won.
This dictionary maps every model's name to its real fitted object, so
whichever one `results_df` says is best is the one actually used and
saved — no silent mismatch between "what we evaluated" and "what we ship."
Now covers all 10 models: ARIMA, SARIMA, Holt-Winters, Prophet, and LSTM
alongside the original ML models.



In [ ]:

fitted_models = {
    "Naive": None,  # not a real model object — has no .predict() to reuse
    "Linear Regression": lr,
    "Random Forest": rf,
    "Random Forest (Tuned)": rf_best,
    "ARIMA": arima_fit,
    "ARIMA (Tuned)": arima_best,
    "SARIMA": sarima_fit,
    "Holt-Winters": hw_fit,
    "Gradient Boosting": gb,
    "XGBoost": xgb,
    "XGBoost (Tuned)": xgb_best,
    "SVR": svr,
    "LSTM": lstm,
    "Prophet": prophet_model,
}
best_model = fitted_models[best_model_name]
print(f"\n>>> Using '{best_model_name}' as the deployed model (matches results_df winner) <<<\n")



18. FEATURE IMPORTANCE / COEFFICIENTS — explains WHY the model predicts

Tree-based models expose .feature_importances_; Linear Regression exposes
.coef_ instead (the weight it puts on each feature). Handle both so this
section works no matter which model actually won.


In [ ]:

if hasattr(best_model, "feature_importances_"):
    importances = pd.DataFrame({
        'Feature': feature_cols,
        'Importance': best_model.feature_importances_
    }).sort_values('Importance', ascending=False)
    plt.figure(figsize=(8, 5))
    sns.barplot(x='Importance', y='Feature', data=importances, color='teal')
    plt.title(f'Feature Importance — {best_model_name}')
    plt.show()
    print(importances)
elif hasattr(best_model, "coef_"):
    coefs = pd.DataFrame({
        'Feature': feature_cols,
        'Coefficient': best_model.coef_
    }).sort_values('Coefficient', key=abs, ascending=False)
    plt.figure(figsize=(8, 5))
    sns.barplot(x='Coefficient', y='Feature', data=coefs, color='teal')
    plt.title(f'Feature Coefficients — {best_model_name}')
    plt.show()
    print(coefs)
else:
    print(f"{best_model_name} doesn't expose feature importances/coefficients directly.")



19. RESIDUAL ANALYSIS — where is the winning model actually wrong?

Rebuild the winning model's test-set predictions cleanly (works whether it
came from sklearn-style X_test or ARIMA's own forecast() call).


In [ ]:

if best_model_name == "ARIMA (Tuned)":
    best_pred = arima_best_pred.values
elif best_model_name == "ARIMA":
    best_pred = arima_pred.values
elif best_model_name == "SARIMA":
    best_pred = sarima_pred.values
elif best_model_name == "Holt-Winters":
    best_pred = hw_pred.values
elif best_model_name == "Prophet":
    best_pred = prophet_pred
elif best_model_name == "LSTM":
    best_pred = lstm_pred
    y_test_for_resid = y_seq_test_actual
else:
    best_pred = best_model.predict(X_test_scaled if best_model_name == "SVR" else X_test)

y_test_for_resid = y_seq_test_actual if best_model_name == "LSTM" else y_test.values
residuals = y_test_for_resid - best_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(test['Date'].values[-len(residuals):], residuals, color='crimson')
axes[0].axhline(0, color='black', linestyle='--')
axes[0].set_title(f'Residuals Over Time — {best_model_name}')
axes[0].set_xlabel('Date'); axes[0].set_ylabel('Actual - Predicted')
sns.histplot(residuals, bins=20, kde=True, color='crimson', ax=axes[1])
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.show()



20. FINAL 30-DAY FORECAST — using the TRUE winning model

Three different forecasting mechanics depending on model family:
  - Feature-based ML models (Linear Regression, RF, GB, XGBoost, SVR):
    recursive forecasting — each day's prediction becomes the input lag
    for the next day's prediction, since there's no real future data.
    IMPORTANT CAVEAT if a tree-based model (RF/GB/XGBoost) wins: tree
    models can't extrapolate beyond the value range seen in training, so
    once the recursive inputs drift outside that range, every subsequent
    day collapses to the same repeated value — this actually happened
    when the original buggy script forecast with XGBoost (flat 391.79
    repeats). Linear Regression doesn't have this problem since it
    extrapolates linearly (though see the coefficient caveat above).
  - Native time-series models (ARIMA, ARIMA (Tuned), SARIMA,
    Holt-Winters): each exposes its own `.forecast(steps=...)` that
    projects the horizon directly from its own statistical structure —
    no manual lag-rolling needed.
  - Prophet: takes a `future` dataframe (built with
    `make_future_dataframe`) instead of a `steps` count, and returns a
    full dataframe of columns including `yhat`.

NOTE: LSTM forecasting is intentionally NOT implemented here — it would
need its own recursive sequence-feeding loop (feed each prediction back
into the last 10-step window). This is fine as long as LSTM isn't the
deployed model; the branch below raises a clear error instead of silently
producing a wrong forecast if LSTM ever does win.



In [ ]:

feature_based_models = [
    "Naive", "Linear Regression", "Random Forest", "Random Forest (Tuned)",
    "Gradient Boosting", "XGBoost", "XGBoost (Tuned)", "SVR",
]
native_forecast_models = ["ARIMA", "ARIMA (Tuned)", "SARIMA", "Holt-Winters"]

future_dates = pd.date_range(start=df['Date'].max() + pd.Timedelta(days=1), periods=30, freq='B')

if best_model_name in feature_based_models:
    last_known = df_model.iloc[-1:].copy()
    future_preds = []
    for i in range(30):
        X_future = last_known[feature_cols]
        if best_model_name == "SVR":
            X_future = scaler.transform(X_future)
        pred = best_model.predict(X_future)[0]
        future_preds.append(pred)

        new_row = last_known.copy()
        new_row['Close_lag_10'] = new_row['Close_lag_5']
        new_row['Close_lag_5'] = new_row['Close_lag_3']
        new_row['Close_lag_3'] = new_row['Close_lag_2']
        new_row['Close_lag_2'] = new_row['Close_lag_1']
        new_row['Close_lag_1'] = pred
        last_known = new_row

    forecast_df = pd.DataFrame({'Date': future_dates, 'Forecast_Close': future_preds})

elif best_model_name in native_forecast_models:
    forecast_values = np.asarray(best_model.forecast(steps=30))
    forecast_df = pd.DataFrame({'Date': future_dates, 'Forecast_Close': forecast_values})

elif best_model_name == "Prophet":
    prophet_future = best_model.make_future_dataframe(periods=30, freq='B')
    prophet_result = best_model.predict(prophet_future)
    forecast_df = pd.DataFrame({
        'Date': future_dates,
        'Forecast_Close': prophet_result['yhat'].iloc[-30:].values,
    })

elif best_model_name == "LSTM":
    raise NotImplementedError(
        "LSTM won the comparison, but the recursive multi-step forecasting "
        "loop for LSTM isn't implemented in this script. Add a sliding-"
        "window loop (predict next day -> append -> drop oldest -> repeat) "
        "before running this cell, or exclude LSTM from best-model selection."
    )

print(forecast_df)



### 95% confidence band, sized from the winning model's own residuals



In [ ]:

residual_std = residuals.std()
forecast_df['Lower_Bound'] = forecast_df['Forecast_Close'] - 1.96 * residual_std
forecast_df['Upper_Bound'] = forecast_df['Forecast_Close'] + 1.96 * residual_std

plt.figure(figsize=(14, 6))
plt.plot(df['Date'].tail(90), df['Close'].tail(90), label='Historical', color='blue')
plt.plot(forecast_df['Date'], forecast_df['Forecast_Close'], label=f'Forecast ({best_model_name})', color='red', linestyle='--')
plt.fill_between(forecast_df['Date'], forecast_df['Lower_Bound'], forecast_df['Upper_Bound'],
                  color='red', alpha=0.15, label='95% Confidence Band')
plt.title('Reliance Industries - 30 Day Forecast with Confidence Interval')
plt.xlabel('Date'); plt.ylabel('Close Price')
plt.legend()
plt.show()

forecast_df.to_csv('Reliance_30_Day_Forecast.csv', index=False)
print("Saved forecast CSV!")



21. SAVE THE TRUE WINNING MODEL (not hardcoded XGBoost)



In [ ]:

import joblib

joblib.dump(best_model, 'reliance_forecast_model.pkl')
print(f"Saved '{best_model_name}' as reliance_forecast_model.pkl")



Sanity check: reload and confirm the saved file reproduces the same score.
Only feature-based sklearn-style models expose .predict(X_test) the same
way after reload — ARIMA/SARIMA/Holt-Winters/Prophet/LSTM don't take a
flat feature matrix, so they're skipped here rather than falsely errored.


In [ ]:
loaded_model = joblib.load('reliance_forecast_model.pkl')
if best_model_name in feature_based_models:
    check_X = X_test_scaled if best_model_name == "SVR" else X_test
    test_pred = loaded_model.predict(check_X)
    evaluate(y_test, test_pred, "Reloaded Model (sanity check)")
else:
    print(f"Skipping reload sanity check — '{best_model_name}' isn't a flat-feature sklearn-style model.")



Only needed if the winning model is SVR — it requires the same
StandardScaler used at training time to transform any new input.



In [ ]:

if best_model_name == "SVR":
    joblib.dump(scaler, 'feature_scaler.pkl')
    print("Saved feature_scaler.pkl (required for SVR)")



22. SAVE DEPLOYMENT METADATA — what was trained, what won, what got saved

The Streamlit app (app.py) reads this JSON to show, in the UI, which
models were trained and compared, which one won, and which file is
actually deployed — instead of guessing from the pickle's Python type.


In [ ]:

import json

model_metadata = {
    "trained_models": list(fitted_models.keys()),  # every model trained & compared
    "best_model": best_model_name,                 # winner by lowest RMSE on the test set
    "best_model_metrics": {
        "RMSE": round(float(results_df.iloc[0]['RMSE']), 4),
        "MAE": round(float(results_df.iloc[0]['MAE']), 4),
        "MAPE": round(float(results_df.iloc[0]['MAPE']), 4),
    },
    "leaderboard": results_df.round(4).to_dict(orient='records'),  # all models, sorted by RMSE
    "saved_model_file": "reliance_forecast_model.pkl",
    "requires_feature_scaler": best_model_name == "SVR",
    "generated_at": pd.Timestamp.today().strftime("%Y-%m-%d %H:%M:%S"),
}

with open('model_metadata.json', 'w') as f:
    json.dump(model_metadata, f, indent=2)

print("Saved model_metadata.json")
print(json.dumps(model_metadata, indent=2))
